# Part 1.1 — Build an agent, the raw way

No framework. No magic. About forty lines of real logic.

Before you reach for LangChain or LangGraph, it is worth seeing what an
"agent" actually *is*. This notebook builds a working shopping assistant
with nothing but the OpenAI SDK and a Python `while` loop.

> **Before you start:** follow [`../setup.md`](../setup.md) and check that
> `python ../llm.py` prints `connection ok`.

## The one idea

An agent is a loop:

1. Send the conversation **and a list of tools** to the model.
2. The model either **answers**, or **asks to call a tool**.
3. If it called a tool: run the function, append the result, go to step 1.
4. If it answered: you are done.

That is the whole thing. Everything else — LangGraph, "ReAct", deep
agents — is a refinement of this loop.

In [ ]:
import json
import sys

sys.path.append("..")  # so we can import llm.py from the repo root
from llm import raw_client, llm_config

import tools  # search_products + get_product_details, from this folder

client = raw_client()
MODEL = llm_config()["model"]
print("Connected. Model:", MODEL)

## Step 1 — Describe the tools to the model

The model cannot see your Python functions. You have to *describe* them:
a name, what they do, and what arguments they take. That description is
just a JSON schema.

Below we describe the two functions in `tools.py`. In the next notebook
LangGraph will generate this schema for you — but writing it once by hand
shows you there is nothing hidden.

In [ ]:
TOOL_SCHEMAS = [
    {
        "type": "function",
        "function": {
            "name": "search_products",
            "description": "Search the product catalogue by keyword.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "Search keywords, e.g. 'warm jacket' or 'backpack'",
                    }
                },
                "required": ["query"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_product_details",
            "description": "Get the full details of one product by its product_id.",
            "parameters": {
                "type": "object",
                "properties": {
                    "product_id": {
                        "type": "string",
                        "description": "The product id, e.g. 'JKT-001'",
                    }
                },
                "required": ["product_id"],
            },
        },
    },
]

# Map each tool name to the real Python function that runs it.
TOOL_FUNCTIONS = {
    "search_products": tools.search_products,
    "get_product_details": tools.get_product_details,
}

print("Described", len(TOOL_SCHEMAS), "tools:", list(TOOL_FUNCTIONS))

## Step 2 — The agent loop

Here is the loop from "The one idea", in code. Read it once — it is the
core of this whole repository.

The system prompt is doing real work: it tells the model to *use the
tools* and **never to invent prices or stock**. That instruction is your
first, weakest guardrail — Part 3 is about turning it into a real one.

In [ ]:
SYSTEM_PROMPT = (
    "You are a shopping assistant for an outdoor-gear shop. "
    "Always use the tools to look up real products before you answer. "
    "Never invent products, prices or stock status — state only what the "
    "tools return. Recommend at most three products and give the price in CHF."
)


def run_agent(user_message, max_steps=6, verbose=True):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_message},
    ]

    for step in range(1, max_steps + 1):
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=TOOL_SCHEMAS,
            temperature=0.1,
        )
        message = response.choices[0].message
        messages.append(message)  # remember what the model said

        # No tool calls -> the model gave its final answer.
        if not message.tool_calls:
            return message.content

        # Otherwise: run each requested tool and feed the result back.
        for call in message.tool_calls:
            name = call.function.name
            args = json.loads(call.function.arguments)
            if verbose:
                print(f"  [step {step}] calling {name}({args})")
            result = TOOL_FUNCTIONS[name](**args)
            messages.append(
                {"role": "tool", "tool_call_id": call.id, "content": result}
            )

    return "Stopped: reached the step limit without a final answer."

## Step 3 — Run it

Ask the agent a real shopping question. Watch the `[step N]` lines: that
is the model deciding, on its own, which tool to call.

In [ ]:
answer = run_agent("I need a warm jacket for winter, budget around CHF 250.")
print()
print(answer)

That is an agent. Roughly forty lines of logic in `run_agent` — a loop, a
list of tool schemas, and a dictionary of functions.

Now try a question that needs **two** tools in sequence: first a search,
then a details lookup.

In [ ]:
answer = run_agent("What is the lightest backpack you sell, and is it in stock?")
print()
print(answer)

## What this loop does *not* do

Be honest about the limits of these forty lines:

- **No memory.** Each call to `run_agent` starts fresh. Ask a follow-up
  with "it" or "those" and the agent has no idea what you mean.
- **No streaming**, no retries, no tracing, no parallel tool calls.
- **The tools are trivial.** `search_products` is a keyword match over 12
  products in a JSON file.

The next notebook rebuilds this exact agent with **LangGraph** — and
memory, streaming and tracing come for free. Seeing the raw loop first is
what makes that worthwhile: you now know what the framework is replacing.

---

**From demo to production.** A keyword search over 12 products is a demo.
A real storefront agent searches tens of thousands of SKUs, where keyword
matching breaks down — a customer asking for a "waterproof shell" never
types the product name "Stormshield Hardshell". That is exactly the
problem **Part 2** solves with retrieval.